In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load dataset
df = pd.read_csv("telecom_customer_churn.csv")

# Standardize column names
df.columns = df.columns.str.strip().str.lower()

print("First 5 rows:")
print(df.head())
print("\nDataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# 2. Remove duplicates and standardize text
df = df.drop_duplicates()

# column content to lowercase
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip().str.lower()

# 3. Convert columns to correct data types
cols = ["seniorcitizen", "tenure", "monthlycharges", "totalcharges"]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print(df.head())

# 4. Remove rows with missing target
df = df.dropna(subset=["churn"])

# 5. Handle missing values
for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    if col not in ["customerid", "churn"] and df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

# 6. Handle outliers using IQR
for col in ["monthlycharges", "totalcharges"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

# 7. Feature engineering
df["tenuregroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 48, float("inf")],
    labels=["new", "medium", "long"]
)

# 8. Export cleaned dataset
df.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)

# 9. Separate features and target
X = df.drop(columns=["customerid", "churn"])
X = pd.get_dummies(X, drop_first=True, dtype=int)
y = df["churn"].map({"yes": 1, "no": 0})

# 10. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 11. Scale numerical features
scale_cols = ["tenure", "monthlycharges", "totalcharges"]
scaler = StandardScaler()

X_train = X_train.copy()
X_test = X_test.copy()

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

# 12. Export encoded dataset
encoded = X.copy()
encoded["churn"] = y
encoded.to_csv("Telecom_Customer_Churn_Encoded.csv", index=False)

# Results
print("\nCleaned dataset shape:", df.shape)
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
print("\nChurn distribution:\n", df["churn"].value_counts())
print("\nMissing values after cleaning:", df.isnull().sum().sum())
print("\nBoth datasets exported successfully!")

First 5 rows:
   customerid  gender  seniorcitizen partner dependents  tenure phoneservice  \
0  7590-VHVEG     NaN              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      multiplelines internetservice onlinesecurity  ... deviceprotection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            NaN  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  techsupport streamingtv streamingmovies       

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df=pd.read_csv("telecom_customer_churn.csv")
df.head()
print(df.dtypes)
print(df.isnull().sum())
print(df.describe())

df.drop_duplicates()

df.columns=df.columns.str.strip().str.lower()
print(df.columns)

df=df.dropna()

for col in df.select_dtypes(include='object').columns:
    df[col]=df[col].str.strip().str.lower()

print(df.head())

cols=["seniorcitizen","tenure","monthlycharges","totalcharges"]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print(df.head())

df = df.dropna(subset=["churn"])

for col in df.select_dtypes(include='object').columns:
    if col not in ["customerid", "churn"]:
        df[col]=df[col].fillna(df[col].mode()[0])

for col in df.select_dtypes(include='number').columns:
    df[col]=df[col].fillna(df[col].median())

df.head()

for col in ['monthlycharges','tenure']:
    q1=df[col].quantile(0.25)
    q3=df[col].quantile(0.75)
    iqr=q3-q1
    df[col]=df[col].clip(q1-1.5*iqr,q3+1.58*iqr)

df['tenuregroup']=pd.cut(
    df['tenure'],bins=[-1,12,48,float('inf')],
    labels=["new","medium","long"]

)
print(df['tenuregroup'].head())

X=df.drop(df['customerid'],df['churn'])



customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object
customerID          0
gender              1
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      1
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
Mo

Data Wrangling
Problem Statement: Data Wrangling on Real Estate Market
Dataset: &quot;RealEstate_Prices.csv&quot;
Description: The dataset contains information about housing prices in
a specific real estate market. It includes various attributes such as
property characteristics, location, sale prices, and other relevant
features. The goal is to perform data wrangling to gain insights into
the factors influencing housing prices and prepare the dataset for
further analysis or modeling.
Tasks to Perform: 1. Import the &quot;RealEstate_Prices.csv&quot; dataset. Clean
column names by removing spaces, special characters, or renaming
them for clarity. 2. Handle missing values in the dataset, deciding on
an appropriate strategy (e.g., imputation or removal). 3. Perform data
merging if additional datasets with relevant information are available

(e.g., neighborhood demographics or nearby amenities). 4. Filter and
subset the data based on specific criteria, such as a particular time
period, property type, or location. 5. Handle categorical variables by
encoding them appropriately (e.g., one-hot encoding or label
encoding) for further analysis. 6. Aggregate the data to calculate
summary statistics or derived metrics such as average sale prices by
neighborhood or property type. 7. Identify and handle outliers or
extreme values in the data that may affect the analysis or modeling
process

data : Property ID,Property Type,Neighborhood,Bedrooms,Bathrooms,Area (sq ft),Year Built,Sale Price ($),Sale Year
1001,Villa,Hillview,3,2.0,1743,2018,586261,2022
1002,Apartment,Hillview,4,4.0,1592,2011,455540,2023
1003,Apartment,Riverside,1,3.0,3618,2003,292714,2021
1004,Apartment,Suburb,4,3.0,1508,2002,831899,2023
1005,Villa,Hillview,3,2.0,2641,2011,383876,2023
code : **Name:** Avanish Suryavanshi

**Roll no.:** BEBD23280
## CL-1 Lab Assignment 3 (Group B): Data Wrangling
### Real Estate Data Preprocessing and Analysis
import pandas as pd
import numpy as np
df = pd.read_csv("RealEstate_Prices.csv")

print("Original Dataset:")
display(df.head())

print("\nDataset Shape:")
print(df.shape)
### 2. Clean Column Names and Handle Missing Values
# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace(r"[^\w]", "", regex=True)
)

print("\nCleaned Column Names:")
print(df.columns.tolist())

print("\nMissing Values Before Handling:")
print(df.isnull().sum())

# Fill numerical missing values with median
df["bathrooms"] = df["bathrooms"].fillna(df["bathrooms"].median())

# Fill categorical missing values with mode
df["neighborhood"] = df["neighborhood"].fillna(df["neighborhood"].mode()[0])

print("\nMissing Values After Handling:")
print(df.isnull().sum())
### 3. Data Merging
# Example additional neighborhood dataset
neighborhood_data = pd.DataFrame({
    "neighborhood": ["Downtown", "Suburb", "Riverside", "Hillview", "Lakeside"],
    "avg_income": [75000, 65000, 60000, 70000, 72000]
})

# Merge the datasets
df = pd.merge(df, neighborhood_data, on="neighborhood", how="left")

print("\nDataset After Merging:")
display(df.head())
### 4. Filter and Subset Data
# Properties sold between 2021 and 2023
filtered_data = df[(df["sale_year"] >= 2021) & (df["sale_year"] <= 2023)]

print("\nProperties Sold Between 2021 and 2023:")
display(filtered_data.head())

# Select only houses
houses = df[df["property_type"] == "House"]
print("\nHouse Properties:")
display(houses.head())

# Select important columns
subset = df[["property_type", "neighborhood", "bedrooms", "area_sq_ft", "sale_price_"]]
print("\nSelected Columns:")
display(subset.head())
### 5. Handle Categorical Variables
# One-hot encoding
df_encoded = pd.get_dummies(
    df,
    columns=["property_type", "neighborhood"],
    dtype=int
)

print("\nDataset After One-Hot Encoding:")
display(df_encoded.head())
print("\nEncoded Columns:")
print(df_encoded.columns.tolist())
### 6. Aggregate Data
# Average sale price by neighborhood
avg_price_neighborhood = (
    df.groupby("neighborhood")["sale_price_"]
    .mean()
    .sort_values(ascending=False)
)
print("\nAverage Sale Price by Neighborhood:")
print(avg_price_neighborhood)

# Average sale price by property type
avg_price_type = (
    df.groupby("property_type")["sale_price_"]
    .mean()
    .sort_values(ascending=False)
)
print("\nAverage Sale Price by Property Type:")
print(avg_price_type)

# Average price per square foot
df["price_per_sq_ft"] = df["sale_price_"] / df["area_sq_ft"]
print("\nPrice Per Square Foot:")
display(df[["property_id", "price_per_sq_ft"]].head())
### 7. Identify and Handle Outliers
# Function to detect outliers using IQR
def detect_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Detect sale price outliers
price_outliers, lower, upper = detect_outliers(df, "sale_price_")
print("\nSale Price Outliers:")
display(price_outliers[["property_id", "sale_price_"]])

print("\nSale Price Bounds:")
print("Lower Bound:", lower)
print("Upper Bound:", upper)

# Detect area outliers
area_outliers, lower_area, upper_area = detect_outliers(df, "area_sq_ft")
print("\nArea Outliers:")
display(area_outliers[["property_id", "area_sq_ft"]])
# Remove outliers
df_clean = df[
    (df["sale_price_"] >= lower) &
    (df["sale_price_"] <= upper) &
    (df["area_sq_ft"] >= lower_area) &
    (df["area_sq_ft"] <= upper_area)
]

print("\nDataset Shape Before Removing Outliers:")
print(df.shape)
print("\nDataset Shape After Removing Outliers:")
print(df_clean.shape)
### Final Dataset
print("\nFinal Cleaned Dataset:")
display(df_clean.head())

print("\nFinal Dataset Information:")
df_clean.info()